# SPICE — Phase 1 Pre-train

> SPICE: Sequence-Protein Interaction under Conditional Environments
> This notebook runs the Pre-train with **TensorFlow**: dynamic Transformer + AdaLN + Head A (Cα coords) + a distogram head, supervised by binned distogram cross-entropy.

Run order: 1️⃣ install deps→ 2️⃣ build TFRecord → 3️⃣ train → 4️⃣ visualize.

In [ ]:
# ① Install dependencies (China pip mirror) + check/enable GPU
# 版本钉死：与保存 checkpoint 的环境一致（TF 2.21 + Keras 3.15）。
# 不同 Keras 版本的优化器 checkpoint 布局/dtype 不同（step_counter int64→float32 等），
# 跨平台续训会 RestoreV2 报错；钉死版本后 optimizer 状态即可完整恢复。
# 注意：Kaggle 上清华镜像可能不通——跑不动就把 "-i https://pypi.tuna.tsinghua.edu.cn/simple" 去掉用默认 PyPI。
!pip install -q -i https://pypi.tuna.tsinghua.edu.cn/simple "tensorflow==2.21" "keras==3.15" datasets huggingface_hub pyarrow polars pyyaml tqdm

import tensorflow as tf
import keras

# HF download endpoint is chosen automatically (handled in dataset.py; do NOT set HF_ENDPOINT here):
#   - Colab: the VM runs on Google Cloud, so the official huggingface.co is reachable directly
#     (pointing at the China mirror would actually fail to download files)
#   - Local dev (in China): use data.hf_endpoint in configs/pretrain.yaml (hf-mirror.com)
from spice_pre.config import load_config
from spice_pre.keras_utils import setup_gpu

cfg = load_config("configs/pretrain.yaml")
setup_gpu(cfg.train.use_gpu, cfg.train.gpu_mem_growth, cfg.train.gpu_devices)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", "ON" if cfg.train.use_gpu else "OFF (forced CPU)")
print("Available devices:", gpus if gpus else ["CPU"])

## 2. Build TFRecord

Downloads the `SPICE-Protein/spice_protein` parquet files from HF, cleans them (keeping only structures with environment labels and valid lengths), then writes TFRecords.

> For debugging use `max_shards=2` to run through quickly; for the real training set it to `0` (all shards, ~1.4 GB download).

In [ ]:
from spice_pre.config import load_config
from spice_pre.data.dataset import build_tfrecords

cfg = load_config("configs/pretrain.yaml")
cfg.data.max_shards = 0          # production: all shards (use_env_filtered=false uses the full ~45k; ~1.4 GB download)
cfg.data.use_env_filtered = False

n = build_tfrecords(cfg)
print("TFRecord record count:", n)

> **🔴 精度（2026-08-13 决定性实验）**：**`use_mixed_precision` 必须 = false（模型全 fp32）** + `distogram_fp16: true`（einsum 走 tensor core）。fp16 混合精度会把模型"冻住"——梯度在深层 fp16 backprop 下溢到 0（grad_norm=0），30 epoch 跑完 rmsd 卡 150-195Å、distogram CE 卡 52。E 组合（fp32 模型 + fp16 einsum）本地验证收链 2.3Å。改 `configs/pretrain.yaml` 的 `use_mixed_precision: false` 后重新上传即可（重训前删 `data/train_cache.*`）。

> **🔴 长度门控（2026-08-13 决定性）**：`coord_max_len: 200`——frame 坐标损失只给 ≤200aa 的链；长链（200-400aa 占全数据 65%）cumsum 误差累积会毒化 encoder，但距离任务在长链上可学 → **distogram 全 45k 学、坐标监督只给短链**（`max_seq_len: 512` 不丢数据）。⚠️ **判断标准 = 看 `ce`，别看 `rmsd`**：门控后长链坐标不监督 → **`rmsd` 会一直高（正常！）**；成败看 `ce` 一路跌破 ~4 并降到 2-3。**别用 rmsd 判断，别中途杀**。

> **Disk cache first (3a):** build the bucketed dataset to disk (`data/train_cache.*`) once *before* training. Training then reads the file each epoch — no mid-training cache stall (the fake 47 s/step first-epoch), no RAM blow-up (fixes Kaggle RAM OOM). Delete the cache if you change `batch_size`/bucketing.
>
> Debug: `epochs=2, max_steps=300` finishes in a few minutes; production: `epochs=30, max_steps=0`.

In [ ]:
# 3a. 预构建训练集落盘 cache（一次性，训练开始前跑；幂等，已存在则跳过）
# 把分桶后的数据集完整流式写盘到 data/train_cache.*：
#   - 训练时每 epoch 直接读文件，不再中途卡 cache 构建（消除首轮 47s/step 假象）
#   - 不把整份数据集囤进 RAM（解决 Kaggle RAM OOM）
# ⚠️ 若改过 batch_size / 分桶边界 / 数据，先删 data/train_cache.* 再重建。
from spice_pre.train_pretrain import build_train_cache

build_train_cache(cfg)   # 首次完整落盘；已存在则直接复用


In [ ]:
from spice_pre.train_pretrain import train

cfg.train.epochs = 30
cfg.train.max_steps = 0          # production: run all epochs

# 🔴 fp16 混合精度会冻结训练（2026-08-13 决定性实验，E/F/G 网格）：
#    梯度在深层 fp16 backprop 下溢到 0（grad_norm=0，30 epoch 跑完 rmsd 卡 150Å、distogram CE 卡 52）。
#    修复 = 模型全 fp32：
cfg.train.use_mixed_precision = False
#    distogram einsum 仍走 fp16（tensor core 加速）——fp32 模型 + fp16 einsum 本地已验证收链 2.3Å，安全：
cfg.model.distogram_fp16 = True

# pair_weight 前 3000 步从 0 线性升到 1.0（暖身）：CE 先学距离图、坐标后精修，
# 防随机游走坐标梯度（rmsd 上百Å）压制 CE。train() 内部若发现 cache 缺失也会自动预构建
# （3a 已建则直接复用）；盯 rmsd 是否随 epoch 下降（目标是 <10Å，理想 <5Å）。
train(cfg)


## 4. Visualize

Loads the best weights and compares the predicted Cα backbone against the ground-truth Cα backbone on one validation sample.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from spice_pre.config import load_config
from spice_pre.models import SPICEPretrainModel
from spice_pre.data.dataset import load_tfrecord_dataset

cfg = load_config("configs/pretrain.yaml")

weights_path = "checkpoints/pretrain/best_weights.weights.h5"
if not os.path.exists(weights_path):
    print("No best-weights file yet. Run step 4 (training) first (best weights are only saved when a validation set exists).")
    raise SystemExit(0)

model = SPICEPretrainModel(cfg.model)
model.load_weights(weights_path)

ds = load_tfrecord_dataset(cfg, "val").take(1)
for x, _ in ds:
    inputs = {"tokens": x["tokens"][None], "env": x["env"][None], "mask": x["mask"][None]}
    out = model(inputs, training=False)
    n = int(tf.reduce_sum(x["mask"]).numpy())
    pred = out["coords"][0, :n].numpy()
    true = x["coords"][:n].numpy()
    L = min(n, 80)
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(true[:L, 0], true[:L, 1], true[:L, 2], "o-", lw=1, label="GT")
    ax.plot(pred[:L, 0], pred[:L, 1], pred[:L, 2], "x--", lw=1, label="Pred")
    ax.legend()
    ax.set_title(f"Cα backbone comparison (first {L} residues)")
    plt.show()
print("Done ✅")

## Next steps

- Full training: `max_shards=0`, `epochs=30`.
- Training curves: `tensorboard --logdir runs/pretrain`.
- Phase 2 (RL): reuse the same trunk, adding Head B/B'/C/D with Rust-engine feedback.